In [6]:

# To develop 2 agents each of completely different sources of information and routing the user query to its agent and get the final answer

# ============================================
# Supervisor Agent with HR and IT Agents
# LangGraph Full Example
# ============================================

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model


# ============================================
# 1. Define State
# ============================================

class AgentState(TypedDict):
    query: str
    selected_agent: str
    answer: str


# ============================================
# 2. Initialize LLM
# ============================================

llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0
)


# ============================================
# 3. Supervisor Node
# Decides which agent to use
# ============================================

def supervisor_node(state: AgentState) -> AgentState:

    query = state["query"].lower()

    if "salary" in query or "leave" in query or "holiday" in query:
        state["selected_agent"] = "hr_agent"

    elif "vpn" in query or "password" in query or "internet" in query:
        state["selected_agent"] = "it_agent"

    else:
        state["selected_agent"] = "it_agent"

    return state


# ============================================
# 4. HR Agent Node
# ============================================

def hr_agent_node(state: AgentState) -> AgentState:

    query = state["query"]

    prompt = f"""
    You are an HR support assistant.
    Answer the employee question clearly.

    Question: {query}
    """

    response = llm.invoke(prompt)

    state["answer"] = response.content

    return state


# ============================================
# 5. IT Agent Node
# ============================================

def it_agent_node(state: AgentState) -> AgentState:

    query = state["query"]

    prompt = f"""
    You are an IT support assistant.
    Help fix the technical issue.

    Question: {query}
    """

    response = llm.invoke(prompt)

    state["answer"] = response.content

    return state


# ============================================
# 6. Routing Function
# ============================================

def router(state: AgentState):

    if state["selected_agent"] == "hr_agent":
        return "hr_agent"

    else:
        return "it_agent"


# ============================================
# 7. Build Graph
# ============================================

graph = StateGraph(AgentState)

graph.add_node("supervisor", supervisor_node)
graph.add_node("hr_agent", hr_agent_node)
graph.add_node("it_agent", it_agent_node)

graph.add_edge(START, "supervisor")

graph.add_conditional_edges(
    "supervisor",
    router,
    {
        "hr_agent": "hr_agent",
        "it_agent": "it_agent"
    }
)

graph.add_edge("hr_agent", END)
graph.add_edge("it_agent", END)


# Compile graph
app = graph.compile()


# ============================================
# 8. Run Examples
# ============================================

# Example 1 — HR Query
result1 = app.invoke({
    "query": "How many leave days do I have?"
})

print("\nHR Agent Response:")
print(result1["answer"])


# Example 2 — IT Query
result2 = app.invoke({
    "query": "My VPN is not working"
})

print("\nIT Agent Response:")
print(result2["answer"])




E:\EDU_CARE\arg_venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
E:\EDU_CARE\arg_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



HR Agent Response:
To check your leave balance, I'll need to look up your employee information in our HR system. Can you please provide me with your employee ID or your name so I can verify your details?

Once I have that information, I can check your leave balance and let you know how many leave days you have available.

IT Agent Response:
I'd be happy to help you troubleshoot the issue with your VPN. Can you please provide me with some more information about the problem you're experiencing? 

Here are some questions to help me narrow down the issue:

1. What type of VPN are you using (e.g. Cisco AnyConnect, OpenVPN, etc.)?
2. Have you recently made any changes to your network or VPN settings?
3. Are you getting any error messages when you try to connect to the VPN?
4. Can you connect to the VPN from a different device or location?
5. Have you tried restarting your computer or router?

Please provide me with as much detail as possible, and I'll do my best to assist you in resolving t